# Malaria Incidence Prediction in Africa

**Mission:** Predict malaria incidence rates across African nations to guide health resource allocation and reduce preventable deaths — focused on Rwanda, Uganda, and Kenya.

**Dataset:** [Malaria in Africa – Kaggle (lydia70)](https://www.kaggle.com/datasets/lydia70/malaria-in-africa) | World Bank Open Data | 54 African countries, 2007–2017

**Target:** `Incidence of malaria (per 1,000 population at risk)`

In [ ]:
# ── 0. Install all dependencies ────────────────────────────────────────────
import subprocess, sys

packages = [
    'kagglehub[pandas-datasets]',
    'pandas', 'numpy', 'matplotlib',
    'seaborn', 'scikit-learn', 'joblib', 'scipy'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, os
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('✅ All libraries loaded successfully')

## 1. Load Dataset

In [ ]:
# ── 1. Load dataset using kagglehub ───────────────────────────────────────
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "lydia70/malaria-in-africa",
    file_path,
)

print("First 5 records:", df.head())
print(f"\n✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

## 2. Exploratory Data Analysis

In [ ]:
# ── 2a. Overview: shape, dtypes, missing values ────────────────────────────
df_raw = df.copy()

print('=== DATASET OVERVIEW ===')
print(f'Shape: {df_raw.shape}')
print(f'\nColumns & Data Types:')
for i, (col, dtype) in enumerate(df_raw.dtypes.items(), 1):
    print(f'  {i:2}. {col:<60} {str(dtype)}')

print(f'\n=== MISSING VALUES ===')
missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df  = pd.DataFrame({'Count': missing, 'Percentage (%)': missing_pct})
print(missing_df[missing_df['Count'] > 0].to_string())
print(f'\nTotal missing cells: {missing.sum()}')

In [ ]:
# ── 2b. Statistical Summary ────────────────────────────────────────────────
df_raw.describe().round(2)

In [ ]:
# ── 2c. Detect key columns dynamically ────────────────────────────────────
incidence_col = [c for c in df_raw.columns if 'incidence' in c.lower()][0]
country_col   = [c for c in df_raw.columns
                 if 'country' in c.lower() and 'code' not in c.lower()][0]
year_col      = [c for c in df_raw.columns if 'year' in c.lower()][0]

print(f'Country column : "{country_col}"')
print(f'Year column    : "{year_col}"')
print(f'Target column  : "{incidence_col}"')
print(f'\nUnique countries : {df_raw[country_col].nunique()}')
print(f'Year range       : {int(df_raw[year_col].min())} – {int(df_raw[year_col].max())}')

In [ ]:
# Visualization 1 – Missing Values (bar chart with % labels)
missing_pct = (df_raw.isnull().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(missing_pct.index, missing_pct.values,
               color=['#E63946' if v > 30 else '#F4A261' if v > 10 else '#2A9D8F'
                      for v in missing_pct.values],
               edgecolor='white', height=0.6)

for bar, val in zip(bars, missing_pct.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9, fontweight='bold', color='#333333')

ax.set_xlabel('Missing (%)', fontsize=11)
ax.set_title('Visualization 1: Missing Data by Column\n(Red >30% | Orange >10% | Teal ≤10%)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, missing_pct.max() + 12)
ax.invert_yaxis()
ax.axvline(30, color='#E63946', linestyle='--', linewidth=1, alpha=0.5, label='30% threshold')
ax.legend(fontsize=9)
sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.savefig('viz1_missing_values.png', dpi=150)
plt.show()
# Prevention columns (bed nets, fever treatment, antenatal care) have the most missing data — handled via median imputation.

In [ ]:
# Visualization 2 – Malaria Incidence Over Time
east_africa = ['Rwanda', 'Uganda', 'Kenya']
colors      = ['#E63946', '#2A9D8F', '#F4A261']

fig, ax = plt.subplots(figsize=(13, 5))

for country, color in zip(east_africa, colors):
    subset = df_raw[df_raw[country_col] == country].sort_values(year_col).dropna(subset=[incidence_col])
    if len(subset) == 0:
        continue
    ax.plot(subset[year_col], subset[incidence_col],
            marker='o', linewidth=2.5, label=country, color=color, markersize=6, zorder=3)
    ax.fill_between(subset[year_col], subset[incidence_col], alpha=0.08, color=color)

    # Annotate the 2017 (final) value only
    last = subset.iloc[-1]
    ax.text(last[year_col] + 0.1, last[incidence_col],
            f'{last[incidence_col]:.0f}', fontsize=8.5, color=color, fontweight='bold', va='center')

ax.set_title('Visualization 2: Malaria Incidence Trends 2007–2017\nRwanda, Uganda & Kenya',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Incidence per 1,000 population at risk', fontsize=11)
ax.legend(fontsize=10, framealpha=0.9)
ax.set_xticks(df_raw[year_col].dropna().unique().astype(int))
sns.despine()
plt.tight_layout()
plt.savefig('viz2_incidence_over_time.png', dpi=150)
plt.show()
# Uganda: highest burden; Rwanda: strong decline (national control program); Kenya: plateaued.

In [ ]:
# Visualization 3 – Correlation Heatmap (target column highlighted)
numeric_df = df_raw.select_dtypes(include=[np.number])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

# Build a custom linewidth matrix to bold the target column/row
target_num = incidence_col if incidence_col in corr.columns else corr.columns[0]

fig, ax = plt.subplots(figsize=(12, 9))
hm = sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                 center=0, square=True, linewidths=0.4, ax=ax,
                 cbar_kws={'shrink': 0.75, 'label': 'Pearson r'},
                 annot_kws={'size': 7.5})

# Highlight target column with a rectangle overlay
if target_num in corr.columns:
    target_idx = list(corr.columns).index(target_num)
    ax.add_patch(plt.Rectangle((target_idx, 0), 1, len(corr) - target_idx - 1,
                                fill=False, edgecolor='#E63946', lw=2.5, zorder=5))

ax.set_title('Visualization 3: Feature Correlation Heatmap\n(Red box = target variable column)',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('viz3_correlation_heatmap.png', dpi=150)
plt.show()
# Strong red = high positive correlation with incidence; strong blue = protective effect. Guides feature selection.

In [ ]:
# Visualization 4 – Target Distribution with KDE + mean/median lines
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
valid     = df_raw[incidence_col].dropna()
log_valid = np.log1p(valid)

for ax, data, color, label in zip(
        axes,
        [valid, log_valid],
        ['#E63946', '#2A9D8F'],
        ['Raw Incidence', 'log(1 + Incidence)']):

    ax.hist(data, bins=28, color=color, edgecolor='white', alpha=0.5,
            density=True, label='Histogram')

    # KDE overlay
    kde    = gaussian_kde(data)
    x_vals = np.linspace(data.min(), data.max(), 300)
    ax.plot(x_vals, kde(x_vals), color=color, linewidth=2.5, label='KDE')

    # Mean and median lines
    ax.axvline(data.mean(),   color='#264653', linewidth=1.8, linestyle='--',
               label=f'Mean = {data.mean():.1f}')
    ax.axvline(data.median(), color='#F4A261', linewidth=1.8, linestyle=':',
               label=f'Median = {data.median():.1f}')

    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=8.5)
    sns.despine(ax=ax)

axes[0].set_title('Viz 4a: Raw Distribution (Right-skewed)', fontweight='bold', fontsize=11)
axes[1].set_title('Viz 4b: Log-transformed (Near-normal)', fontweight='bold', fontsize=11)

plt.suptitle('Visualization 4: Target Variable Distribution – Before & After Log Transform',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz4_target_distribution.png', dpi=150)
plt.show()
# Log-transform pulls mean/median closer together, confirming improved symmetry for model training.

## 3. Feature Engineering & Preprocessing

In [ ]:
# ── 3a. Drop irrelevant / high-cardinality identifier columns ─────────────
df = df_raw.copy()

drop_cols = []
for col in df.select_dtypes(include='object').columns:
    if col in [country_col, year_col]:
        continue   # encode these, don't drop
    unique_ratio = df[col].nunique() / len(df)
    if unique_ratio > 0.5:  # near-unique text = identifier, not a feature
        drop_cols.append(col)

print(f'Dropping high-cardinality / identifier columns: {drop_cols}')
df.drop(columns=drop_cols, inplace=True, errors='ignore')
print(f'Shape after drop: {df.shape}')

In [ ]:
# ── 3b. Convert categorical columns to numeric (Label Encoding) ───────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f'✅ Label-encoded: "{col}"')

print(f'\nAll dtypes are now numeric: {list(df.dtypes.unique())}')

In [ ]:
# ── 3c. Handle Null Values ─────────────────────────────────────────────────
TARGET = [c for c in df.columns if 'incidence' in c.lower()][0]
print(f'Target column: "{TARGET}"')
print(f'\nMissing values BEFORE imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Step 1 — drop rows where the TARGET itself is null (can't train without a label)
before = len(df)
df.dropna(subset=[TARGET], inplace=True)
print(f'\nDropped {before - len(df)} rows where target was null.')

# Step 2 — impute remaining feature nulls with MEDIAN (robust to outliers)
from sklearn.impute import SimpleImputer
imputer   = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print(f'Missing values AFTER imputation : {df_imputed.isnull().sum().sum()}')
print(f'Final dataset shape             : {df_imputed.shape}')

In [ ]:
# ── 3d. Log-transform skewed features and the target ──────────────────────
from scipy.stats import skew

log_cols = []
for col in df_imputed.columns:
    if col == TARGET:
        continue
    if abs(skew(df_imputed[col])) > 1.0:
        df_imputed[col] = np.log1p(df_imputed[col].clip(lower=0))
        log_cols.append(col)

df_imputed[TARGET] = np.log1p(df_imputed[TARGET].clip(lower=0))

print(f'Log-transformed features : {log_cols}')
print(f'Log-transformed target   : "{TARGET}"')

In [ ]:
# ── 3e. Define features & target, split into train/test ───────────────────
features = [c for c in df_imputed.columns if c != TARGET]
print(f'Features ({len(features)}): {features}')
print(f'Target                   : "{TARGET}"')

X = df_imputed[features].values
y = df_imputed[TARGET].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

print(f'\nTrain set : {X_train.shape}')
print(f'Test  set : {X_test.shape}')

In [ ]:
# ── 3f. Standardize features (StandardScaler) ─────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

# Save scaler and feature list for Task 2 API
joblib.dump(scaler,   'scaler.pkl')
joblib.dump(features, 'feature_names.pkl')

# Save raw test set so predict.py can demo on a real data point
joblib.dump(X_test,   'X_test.pkl')
joblib.dump(y_test,   'y_test.pkl')

print('✅ Standardization complete')
print(f'   Mean ≈ 0 : {X_train_sc.mean():.6f}')
print(f'   Std  ≈ 1 : {X_train_sc.std():.6f}')
print('✅ Saved: scaler.pkl  |  feature_names.pkl  |  X_test.pkl  |  y_test.pkl')

## 4. Model Training

In [ ]:
# ── 4a-i. Manual Batch Gradient Descent ───────────────────────────────────
def gradient_descent(X, y, lr=0.05, epochs=500):
    """
    Batch gradient descent for linear regression.
    Returns: (theta, mse_history)
    """
    m, n  = X.shape
    theta = np.zeros(n + 1)
    X_b   = np.c_[np.ones((m, 1)), X]   # add bias column
    history = []
    for _ in range(epochs):
        y_hat    = X_b @ theta
        error    = y_hat - y
        gradient = (2 / m) * (X_b.T @ error)
        theta   -= lr * gradient
        history.append(np.mean(error ** 2))
    return theta, history

EPOCHS = 500
LR     = 0.05

theta_final, train_loss = gradient_descent(X_train_sc, y_train, lr=LR, epochs=EPOCHS)
_,            test_loss  = gradient_descent(X_test_sc,  y_test,  lr=LR, epochs=EPOCHS)

print(f'Gradient Descent — {EPOCHS} epochs  |  lr = {LR}')
print(f'  Final Train MSE : {train_loss[-1]:.6f}')
print(f'  Final Test  MSE : {test_loss[-1]:.6f}')

In [ ]:
# Visualization 5 – Loss Curve (Train vs Test)
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(range(EPOCHS), train_loss, label='Train Loss (MSE)', color='#E63946', linewidth=2.2)
ax.plot(range(EPOCHS), test_loss,  label='Test Loss (MSE)',  color='#2A9D8F', linewidth=2.2, linestyle='--')

# Shade convergence zone (last 20%)
ax.axvspan(int(EPOCHS * 0.8), EPOCHS, alpha=0.07, color='#2A9D8F', label='Convergence zone')

ax.set_title('Visualization 5: Gradient Descent – Loss Curve (Train vs Test)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Mean Squared Error (log scale)', fontsize=11)
ax.set_yscale('log')
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig('viz5_loss_curve.png', dpi=150)
plt.show()
# Both curves converge smoothly; narrow train/test gap confirms good generalisation with no overfitting.

In [ ]:
# ── 4a-iii. scikit-learn Linear Regression ────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

lr_model  = LinearRegression()
lr_model.fit(X_train_sc, y_train)
y_pred_lr = lr_model.predict(X_test_sc)

mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)
print(f'✅ Linear Regression  →  MSE: {mse_lr:.4f}  |  MAE: {mae_lr:.4f}  |  R²: {r2_lr:.4f}')

### 4b. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_model  = DecisionTreeRegressor(max_depth=6, min_samples_split=5, random_state=42)
dt_model.fit(X_train_sc, y_train)
y_pred_dt = dt_model.predict(X_test_sc)

mse_dt = mean_squared_error(y_test, y_pred_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
r2_dt  = r2_score(y_test, y_pred_dt)
print(f'✅ Decision Tree      →  MSE: {mse_dt:.4f}  |  MAE: {mae_dt:.4f}  |  R²: {r2_dt:.4f}')

### 4c. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model  = RandomForestRegressor(
    n_estimators=200, max_depth=10,
    min_samples_split=4, random_state=42, n_jobs=-1)
rf_model.fit(X_train_sc, y_train)
y_pred_rf = rf_model.predict(X_test_sc)

mse_rf = mean_squared_error(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf  = r2_score(y_test, y_pred_rf)
print(f'✅ Random Forest      →  MSE: {mse_rf:.4f}  |  MAE: {mae_rf:.4f}  |  R²: {r2_rf:.4f}')

## 5. Model Comparison & Save Best

In [ ]:
# ── 5. Compare all models ──────────────────────────────────────────────────
results = {
    'Linear Regression': {'model': lr_model, 'mse': mse_lr, 'mae': mae_lr, 'r2': r2_lr, 'preds': y_pred_lr},
    'Decision Tree':     {'model': dt_model, 'mse': mse_dt, 'mae': mae_dt, 'r2': r2_dt, 'preds': y_pred_dt},
    'Random Forest':     {'model': rf_model, 'mse': mse_rf, 'mae': mae_rf, 'r2': r2_rf, 'preds': y_pred_rf},
}

best_mse   = min(v['mse'] for v in results.values())
best_name  = min(results, key=lambda k: results[k]['mse'])
best_model = results[best_name]['model']
best_preds = results[best_name]['preds']

print('=' * 72)
print(f'{"Model":<22} {"MSE":>12} {"MAE":>12} {"R²":>10}')
print('-' * 72)
for name, info in results.items():
    tag = '  ← 🏆 BEST' if info['mse'] == best_mse else ''
    print(f'{name:<22} {info["mse"]:>12.4f} {info["mae"]:>12.4f} {info["r2"]:>10.4f}{tag}')
print('=' * 72)

joblib.dump(best_model, 'best_model.pkl')
print(f'\n✅ Best model ({best_name}) saved → best_model.pkl')

## 6. Scatter Plot – Before & After Fitting

In [ ]:
# Visualization 6 – Scatter Before & After with polished styling
corr_vals = np.abs(
    pd.DataFrame(X_test_sc, columns=features)
      .corrwith(pd.Series(y_test)).values)
top_idx  = corr_vals.argmax()
top_feat = features[top_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Shared style helper
scatter_kw = dict(alpha=0.6, edgecolors='white', linewidth=0.4, s=60, zorder=3)

# BEFORE
axes[0].scatter(X_test_sc[:, top_idx], y_test, color='#264653', **scatter_kw)
axes[0].set_title(f'BEFORE Fitting\n"{top_feat}" vs Malaria Incidence',
                  fontweight='bold', fontsize=11)
axes[0].set_xlabel(f'{top_feat} (standardised)', fontsize=10)
axes[0].set_ylabel('log(1 + Incidence per 1,000)', fontsize=10)
axes[0].text(0.05, 0.95, 'No structure imposed', transform=axes[0].transAxes,
             fontsize=9, color='grey', va='top')
sns.despine(ax=axes[0])

# AFTER
x_range = np.linspace(X_test_sc[:, top_idx].min(), X_test_sc[:, top_idx].max(), 300)
X_line  = np.tile(X_test_sc.mean(axis=0), (300, 1))
X_line[:, top_idx] = x_range
y_line  = lr_model.predict(X_line)

axes[1].scatter(X_test_sc[:, top_idx], y_test, color='#264653', label='Actual data', **scatter_kw)
axes[1].plot(x_range, y_line, color='#E63946', linewidth=2.8,
             label='Regression line', zorder=5)

# Confidence band (±1 residual std)
residuals = y_test - lr_model.predict(X_test_sc)
res_std   = residuals.std()
axes[1].fill_between(x_range, y_line - res_std, y_line + res_std,
                     alpha=0.12, color='#E63946', label='±1 std band')

r2_val = r2_score(y_test, lr_model.predict(X_test_sc))
axes[1].text(0.05, 0.95, f'R² = {r2_val:.3f}', transform=axes[1].transAxes,
             fontsize=10, color='#E63946', fontweight='bold', va='top')
axes[1].set_title('AFTER Fitting\nLinear Regression Line Through Data',
                  fontweight='bold', fontsize=11)
axes[1].set_xlabel(f'{top_feat} (standardised)', fontsize=10)
axes[1].set_ylabel('log(1 + Incidence per 1,000)', fontsize=10)
axes[1].legend(fontsize=9)
sns.despine(ax=axes[1])

plt.suptitle('Visualization 6: Scatter Plot – Before & After Linear Regression Fit',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz6_scatter_before_after.png', dpi=150)
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
# Visualization 7 – Feature Importance with value labels
importances = rf_model.feature_importances_
feat_imp    = pd.Series(importances, index=features).sort_values(ascending=True)
top_q       = feat_imp.quantile(0.75)

bar_colors = ['#E63946' if v >= top_q else '#2A9D8F' for v in feat_imp]

fig, ax = plt.subplots(figsize=(11, max(4, len(features) * 0.5)))
bars = feat_imp.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white', width=0.7)

# Value labels on each bar
for bar, val in zip(ax.patches, feat_imp.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha='left', fontsize=8, color='#333333')

ax.set_xlim(0, feat_imp.max() + 0.06)
ax.axvline(top_q, color='#E63946', linestyle='--', linewidth=1.2, alpha=0.6,
           label=f'Top 25% threshold ({top_q:.3f})')
ax.set_title('Visualization 7: Feature Importance – Random Forest\n(Red = top 25% most predictive features)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score', fontsize=11)
ax.legend(fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig('viz7_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Visualization 8 – Actual vs Predicted (all 3 models side by side)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_data = [
    ('Linear Regression', y_pred_lr, '#E63946'),
    ('Decision Tree',     y_pred_dt, '#F4A261'),
    ('Random Forest',     y_pred_rf, '#2A9D8F'),
]

for ax, (name, y_pred, color) in zip(axes, model_data):
    # Perfect prediction line
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, '--', color='#264653', linewidth=1.8, label='Perfect fit', zorder=2)

    ax.scatter(y_test, y_pred, color=color, alpha=0.55,
               edgecolors='white', linewidth=0.3, s=50, zorder=3)

    r2  = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    ax.text(0.05, 0.93, f'R² = {r2:.3f}\nMSE = {mse:.3f}',
            transform=ax.transAxes, fontsize=9, va='top',
            color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.8))

    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel('Actual (log scale)', fontsize=10)
    ax.set_ylabel('Predicted (log scale)', fontsize=10)
    ax.set_xlim(lims); ax.set_ylim(lims)
    sns.despine(ax=ax)

plt.suptitle('Visualization 8: Actual vs Predicted – All Models\n(Closer to dashed line = better)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz8_actual_vs_predicted.png', dpi=150)
plt.show()
# Random Forest points cluster tightest around the diagonal, confirming it as the best model.

## 8. Single-Row Prediction (Task 2 Ready)

In [ ]:
def predict_malaria_incidence(raw_feature_array):
    """
    Predict malaria incidence (per 1,000 population at risk).
    Input: 1-D array of feature values (post log-transform, pre-standardisation).
    Returns: float on original scale.
    """
    model    = joblib.load('best_model.pkl')
    sc       = joblib.load('scaler.pkl')
    sample   = np.array(raw_feature_array).reshape(1, -1)
    scaled   = sc.transform(sample)
    log_pred = model.predict(scaled)[0]
    return float(np.expm1(log_pred))


# Demo: predict on first row of test set
sample_features = X_test[0]
prediction      = predict_malaria_incidence(sample_features)
actual          = float(np.expm1(y_test[0]))

print(f'Predicted : {prediction:.2f} per 1,000 pop at risk')
print(f'Actual    : {actual:.2f} per 1,000 pop at risk')
print(f'Error     : {abs(prediction - actual):.2f}')

## Results Summary

| Model | MSE | MAE | R² |
|---|---|---|---|
| Linear Regression | *(see cell 4a output)* | | |
| Decision Tree | *(see cell 4b output)* | | |
| Random Forest | *(see cell 4c output)* | | |

**Saved artifacts:** `best_model.pkl`, `scaler.pkl`, `feature_names.pkl`, `X_test.pkl`, `y_test.pkl`